# DFoT Novel View Synthesis — Single Image → Video

Generate a novel-view video from **one starting image** and a **RealEstate10K camera trajectory**.

Uses the pretrained [DFoT](https://github.com/kwsong0113/diffusion-forcing-transformer) model.

### Steps
1. **Run Section 1** once to set up the environment (≈ 3–5 min on first run)
2. **Edit Section 2** with your image/trajectory paths
3. **Run Section 3** to generate your video

> The pretrained checkpoint (~1 GB) is downloaded automatically on first run and cached in `/content/`.


## Section 1 — Setup (run once per session)

In [ ]:
# Mount Google Drive so you can read input files and save the output video
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/DFoT_Outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Drive mounted.  Outputs → {OUTPUT_DIR}")


In [ ]:
import os, sys

REPO_DIR = '/content/diffusion-forcing-transformer'

# Clone repo (skipped if already present)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --depth 1 https://github.com/kwsong0113/diffusion-forcing-transformer.git {REPO_DIR}')
    print(f"Cloned repo → {REPO_DIR}")
else:
    print(f"Repo already at {REPO_DIR}")

# Install additional dependencies not bundled with Colab
os.system(
    'pip install -q '
    'hydra-core==1.3.2 einops lightning wandb roma imageio diffusers transformers pytubefix'
)
print("Dependencies installed.")


In [ ]:
# Patch the cloned repo with the custom dataset + inference script
import base64, os, pathlib

REPO_DIR = '/content/diffusion-forcing-transformer'

# ── 1. datasets/video/single_image_trajectory.py ──────────────────────────────
DATASET_B64 = (
    "IiIiCkRhdGFzZXQgZm9yIGdlbmVyYXRpbmcgYSB2aWRlbyBmcm9tIGEgc2luZ2xlIGlucHV0IGltYWdlICsgUmVhbEVzdGF0ZTEwSyBjYW1lcmEgdHJhamVjdG9yeS4KClRoZSBkYXRhc2V0IHdyYXBzIGEgdXNlci1wcm92aWRlZCBpbWFnZSBhbmQgYSBSZWFsRXN0YXRlMTBLLWZvcm1hdCAudHh0IHRyYWplY3RvcnkgZmlsZQpzbyB0aGF0IHRoZXkgY2FuIGJlIGZlZCBpbnRvIHRoZSBleGlzdGluZyBERm9UIHZpZGVvLWdlbmVyYXRpb24gcGlwZWxpbmUuCiIiIgoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2h2aXNpb24udHJhbnNmb3Jtcy5mdW5jdGlvbmFsIGFzIFRGCmZyb20gb21lZ2Fjb25mIGltcG9ydCBEaWN0Q29uZmlnCmZyb20gUElMIGltcG9ydCBJbWFnZQoKZnJvbSAuYmFzZV92aWRlbyBpbXBvcnQgU1BMSVQKCgpjbGFzcyBTaW5nbGVJbWFnZVRyYWplY3RvcnlEYXRhc2V0KHRvcmNoLnV0aWxzLmRhdGEuRGF0YXNldCk6CiAgICAiIiIKICAgIEEgbWluaW1hbCBkYXRhc2V0IHRoYXQgd3JhcHMgYSBzaW5nbGUgaW1hZ2UgYW5kIGEgY2FtZXJhIHRyYWplY3RvcnkgLnR4dCBmaWxlLgoKICAgIFRyYWplY3RvcnkgZm9ybWF0IChSZWFsRXN0YXRlMTBLKToKICAgICAgICBMaW5lIDEgIDogWW91VHViZSBVUkwgKGlnbm9yZWQg4oCUIHVzZXIgc3VwcGxpZXMgdGhlaXIgb3duIGltYWdlKQogICAgICAgIExpbmVzIDIrOiA8dGltZXN0YW1wPiA8djA+IDx2MT4gLi4uIDx2MTc+CiAgICAgICAgICAgICAgICAgIDE4IGZsb2F0cyBwZXIgbGluZToKICAgICAgICAgICAgICAgICAgICBbMDo0XSAg4oCTIGludHJpbnNpY3MgZngsIGZ5LCBweCwgcHkgKG5vcm1hbGlzZWQpCiAgICAgICAgICAgICAgICAgICAgWzQ6Nl0gIOKAkyB0d28gdmFsdWVzIHNraXBwZWQgKGludGVybmFsIFJFMTBLIGFydGVmYWN0KQogICAgICAgICAgICAgICAgICAgIFs2OjE4XSDigJMgM8OXNCB3b3JsZC10by1jYW1lcmEgZXh0cmluc2ljIG1hdHJpeCwgcm93LW1ham9yCgogICAgQWZ0ZXIgcHJvY2Vzc2luZywgZWFjaCBmcmFtZSBwcm9kdWNlcyBhIDE2LWRpbSB2ZWN0b3I6CiAgICAgICAgNCBpbnRyaW5zaWNzICsgMTIgZXh0cmluc2ljcyAgKG1hdGNoaW5nIGRhdGFzZXQuZXh0ZXJuYWxfY29uZF9kaW09MTYpCgogICAgVGhlIGRhdGFzZXQgYWx3YXlzIHJldHVybnMgdGhlIHNhbWUgc2FtcGxlICh0aGUgaW5wdXQgaW1hZ2UgKyBzYW1wbGVkIHRyYWplY3RvcnkpLAogICAgcmVwZWF0ZWQgYGNmZy5udW1fc2FtcGxlc2AgdGltZXMgc28gdGhhdCB0aGUgdmFsaWRhdGlvbiBsb29wIGNhbiBnZW5lcmF0ZQogICAgbXVsdGlwbGUgdmlkZW9zIGZyb20gb25lIGNhbGwgaWYgZGVzaXJlZC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjZmc6IERpY3RDb25maWcsIHNwbGl0OiBTUExJVCA9ICJ2YWxpZGF0aW9uIik6CiAgICAgICAgc2VsZi5jZmcgPSBjZmcKICAgICAgICBzZWxmLm5fZnJhbWVzOiBpbnQgPSBjZmcubl9mcmFtZXMKICAgICAgICBzZWxmLnJlc29sdXRpb246IGludCA9IGNmZy5yZXNvbHV0aW9uCiAgICAgICAgc2VsZi5jb250ZXh0X2xlbmd0aDogaW50ID0gY2ZnLmNvbnRleHRfbGVuZ3RoCiAgICAgICAgc2VsZi5udW1fc2FtcGxlczogaW50ID0gY2ZnLmdldCgibnVtX3NhbXBsZXMiLCAxKQoKICAgICAgICBzZWxmLmltYWdlID0gc2VsZi5fbG9hZF9pbWFnZShQYXRoKGNmZy5pbWFnZV9wYXRoKSkKICAgICAgICBzZWxmLnBvc2VzID0gc2VsZi5fcGFyc2VfdHJhamVjdG9yeShQYXRoKGNmZy50cmFqZWN0b3J5X3BhdGgpKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIEludGVybmFsIGhlbHBlcnMKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIF9sb2FkX2ltYWdlKHNlbGYsIHBhdGg6IFBhdGgpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJMb2FkLCBjZW50cmUtY3JvcCB0byBzcXVhcmUsIHJlc2l6ZS4gUmV0dXJucyAoMywgSCwgVykgaW4gWzAsIDFdLiIiIgogICAgICAgIGltZyA9IEltYWdlLm9wZW4ocGF0aCkuY29udmVydCgiUkdCIikKICAgICAgICB3LCBoID0gaW1nLnNpemUKICAgICAgICBtaW5fZGltID0gbWluKHcsIGgpCiAgICAgICAgaW1nID0gaW1nLmNyb3AoKAogICAgICAgICAgICAodyAtIG1pbl9kaW0pIC8vIDIsCiAgICAgICAgICAgIChoIC0gbWluX2RpbSkgLy8gMiwKICAgICAgICAgICAgKHcgKyBtaW5fZGltKSAvLyAyLAogICAgICAgICAgICAoaCArIG1pbl9kaW0pIC8vIDIsCiAgICAgICAgKSkKICAgICAgICBpbWcgPSBpbWcucmVzaXplKChzZWxmLnJlc29sdXRpb24sIHNlbGYucmVzb2x1dGlvbiksIEltYWdlLkxBTkNaT1MpCiAgICAgICAgcmV0dXJuIFRGLnRvX3RlbnNvcihpbWcpICAjICgzLCBILCBXKQoKICAgIGRlZiBfcGFyc2VfdHJhamVjdG9yeShzZWxmLCBwYXRoOiBQYXRoKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgUGFyc2UgdGhlIHRyYWplY3RvcnkgZmlsZSBhbmQgcmV0dXJuIChuX2ZyYW1lcywgMTYpIHByb2Nlc3NlZCBwb3Nlcy4KCiAgICAgICAgSGFuZGxlcyBib3RoIDE4LXZhbHVlIHJhdyBSRTEwSyBmaWxlcyBhbmQgMTYtdmFsdWUgcHJlLXByb2Nlc3NlZCBmaWxlcy4KICAgICAgICAiIiIKICAgICAgICBjYW1lcmFzID0gW10KICAgICAgICB3aXRoIG9wZW4ocGF0aCwgInIiKSBhcyBmOgogICAgICAgICAgICBsaW5lcyA9IGYucmVhZGxpbmVzKCkKCiAgICAgICAgZm9yIGksIGxpbmUgaW4gZW51bWVyYXRlKGxpbmVzKToKICAgICAgICAgICAgaWYgaSA9PSAwOgogICAgICAgICAgICAgICAgY29udGludWUgICMgZmlyc3QgbGluZSBpcyBZb3VUdWJlIFVSTAogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGFydHMgPSBsaW5lLnNwbGl0KCkKICAgICAgICAgICAgY2FtID0gbnAuYXJyYXkoW2Zsb2F0KHgpIGZvciB4IGluIHBhcnRzWzE6XV0sIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgICAgIGNhbWVyYXMuYXBwZW5kKGNhbSkKCiAgICAgICAgaWYgbGVuKGNhbWVyYXMpID09IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJObyBjYW1lcmEgZnJhbWVzIGZvdW5kIGluIHtwYXRofSIpCgogICAgICAgIGNhbWVyYXMgPSBucC5zdGFjayhjYW1lcmFzKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoTiwgcmF3X2RpbSkKICAgICAgICBjYW1lcmFzID0gdG9yY2gudGVuc29yKGNhbWVyYXMsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgTiwgcmF3X2RpbSA9IGNhbWVyYXMuc2hhcGUKCiAgICAgICAgIyBVbmlmb3JtbHkgc3Vic2FtcGxlIC8gaW50ZXJwb2xhdGUgdG8gZXhhY3RseSBuX2ZyYW1lcyBwb3NlcwogICAgICAgIGlmIE4gPj0gc2VsZi5uX2ZyYW1lczoKICAgICAgICAgICAgaW5kaWNlcyA9IHRvcmNoLmxpbnNwYWNlKDAsIE4gLSAxLCBzZWxmLm5fZnJhbWVzKS5yb3VuZCgpLmxvbmcoKQogICAgICAgICAgICBjYW1lcmFzID0gY2FtZXJhc1tpbmRpY2VzXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgTGluZWFyIGludGVycG9sYXRpb24gd2hlbiB0aGUgdHJhamVjdG9yeSBpcyBzaG9ydGVyIHRoYW4gbl9mcmFtZXMKICAgICAgICAgICAgdCA9IHRvcmNoLmxpbnNwYWNlKDAsIE4gLSAxLCBzZWxmLm5fZnJhbWVzKQogICAgICAgICAgICBsbyA9IHQuZmxvb3IoKS5sb25nKCkuY2xhbXAoMCwgTiAtIDIpCiAgICAgICAgICAgIGhpID0gKGxvICsgMSkuY2xhbXAoMCwgTiAtIDEpCiAgICAgICAgICAgIGFscGhhID0gKHQgLSBsby5mbG9hdCgpKS51bnNxdWVlemUoLTEpCiAgICAgICAgICAgIGNhbWVyYXMgPSBjYW1lcmFzW2xvXSAqICgxIC0gYWxwaGEpICsgY2FtZXJhc1toaV0gKiBhbHBoYQoKICAgICAgICAjIE1pcnJvciBfcHJvY2Vzc19leHRlcm5hbF9jb25kKCkgaW4gZGF0YXNldHMvdmlkZW8vcmVhbGVzdGF0ZTEway5weToKICAgICAgICAjICAga2VlcCBpbnRyaW5zaWNzIChjb2xzIDAtMykgYW5kIGV4dHJpbnNpY3MgKGNvbHMgNi0xNyksIHNraXAgY29scyA0LTUKICAgICAgICBpZiByYXdfZGltID09IDE4OgogICAgICAgICAgICBwb3NlcyA9IHRvcmNoLmNhdChbY2FtZXJhc1s6LCA6NF0sIGNhbWVyYXNbOiwgNjpdXSwgZGltPS0xKSAgIyAoVCwgMTYpCiAgICAgICAgZWxpZiByYXdfZGltID09IDE2OgogICAgICAgICAgICBwb3NlcyA9IGNhbWVyYXMgICAgICAgICAgIyBhbHJlYWR5IGluIHByb2Nlc3NlZCBmb3JtYXQKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJFeHBlY3RlZCAxOCAocmF3IFJFMTBLKSBvciAxNiAocHJlLXByb2Nlc3NlZCkgY2FtZXJhIHZhbHVlcyBwZXIgbGluZSwgIgogICAgICAgICAgICAgICAgZiJnb3Qge3Jhd19kaW19IGluIHtwYXRofS4iCiAgICAgICAgICAgICkKCiAgICAgICAgcmV0dXJuIHBvc2VzICAjIChuX2ZyYW1lcywgMTYpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgICMgRGF0YXNldCBpbnRlcmZhY2UKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLm51bV9zYW1wbGVzCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KToKICAgICAgICBUID0gc2VsZi5uX2ZyYW1lcwogICAgICAgICMgRmlsbCBldmVyeSBmcmFtZSB3aXRoIHRoZSBpbnB1dCBpbWFnZTsgdGhlIG1vZGVsIHJlcGxhY2VzIG5vbi1jb250ZXh0IGZyYW1lcy4KICAgICAgICB2aWRlbyA9IHNlbGYuaW1hZ2UudW5zcXVlZXplKDApLmV4cGFuZChULCAtMSwgLTEsIC0xKS5jbG9uZSgpICAjIChULCAzLCBILCBXKQogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ2aWRlb3MiOiB2aWRlbywgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgKFQsIDMsIEgsIFcpIGluIFswLCAxXQogICAgICAgICAgICAiY29uZHMiOiBzZWxmLnBvc2VzLCAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChULCAxNikKICAgICAgICAgICAgIm5vbnRlcm1pbmFsIjogdG9yY2gub25lcyhULCBkdHlwZT10b3JjaC5ib29sKSwgICAjIGFsbCBmcmFtZXMgYXJlIHZhbGlkCiAgICAgICAgfQo="
)
target = pathlib.Path(REPO_DIR) / 'datasets/video/single_image_trajectory.py'
target.write_bytes(base64.b64decode(DATASET_B64))

# ── 2. configurations/dataset/single_image_trajectory.yaml ────────────────────
CONFIG_B64 = (
    "IyBEYXRhc2V0IGNvbmZpZyBmb3IgZ2VuZXJhdGluZyBhIHZpZGVvIGZyb20gYSBzaW5nbGUgaW1hZ2UgKyBSZWFsRXN0YXRlMTBLIHRyYWplY3RvcnkuCiMgQWxsIHZhbHVlcyBhcmUgbWF0Y2hlZCB0byB0aGUgREZvVF9SRTEwSy5ja3B0IHByZXRyYWluZWQgbW9kZWwuCiMKIyBSZXF1aXJlZCBvdmVycmlkZXMgKHBhc3Mgb24gQ0xJIG9yIHZpYSBnZW5lcmF0ZV92aWRlby5weSk6CiMgICBkYXRhc2V0LmltYWdlX3BhdGg9L3BhdGgvdG8vaW1hZ2UuanBnCiMgICBkYXRhc2V0LnRyYWplY3RvcnlfcGF0aD0vcGF0aC90by90cmFqZWN0b3J5LnR4dAoKZGVmYXVsdHM6CiAgLSBiYXNlX3ZpZGVvCgojIOKUgOKUgCBVc2VyLXByb3ZpZGVkIGlucHV0cyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKaW1hZ2VfcGF0aDogPz8/ICAgICAgICAgIyAocmVxdWlyZWQpIHBhdGggdG8gdGhlIHN0YXJ0aW5nIGltYWdlCnRyYWplY3RvcnlfcGF0aDogPz8/ICAgICMgKHJlcXVpcmVkKSBwYXRoIHRvIGEgUmVhbEVzdGF0ZTEwSy1mb3JtYXQgLnR4dCB0cmFqZWN0b3J5CgojIOKUgOKUgCBNdXN0IG1hdGNoIHRoZSBwcmV0cmFpbmVkIERGb1RfUkUxMEsgY2hlY2twb2ludCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKcmVzb2x1dGlvbjogMjU2Cm1heF9mcmFtZXM6IDggICAgICAgICAgICMgY29udGV4dC13aW5kb3cgc2l6ZSB0aGUgbW9kZWwgd2FzIHRyYWluZWQgd2l0aApleHRlcm5hbF9jb25kX2RpbTogMTYgICAjIDQgaW50cmluc2ljcyArIDEyIGV4dHJpbnNpY3MgcGVyIGZyYW1lCmV4dGVybmFsX2NvbmRfc3RhY2s6IGZhbHNlCmV4dGVybmFsX2NvbmRfcHJvY2Vzc2luZzogbnVsbAoKIyDilIDilIAgUkUxMEsgbm9ybWFsaXNhdGlvbiBjb25zdGFudHMgKGZyb20gdGhlIHRyYWluaW5nIHNldCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACmRhdGFfbWVhbjogW1tbMC41NzddXSwgW1swLjUxN11dLCBbWzAuNDYxXV1dCmRhdGFfc3RkOiAgW1tbMC4yNDldXSwgW1swLjI0OV1dLCBbWzAuMjY4XV1dCgojIOKUgOKUgCBJbmZlcmVuY2Ugc2V0dGluZ3MgKGNhbiBiZSBvdmVycmlkZGVuIG9uIHRoZSBDTEkpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApuX2ZyYW1lczogNTAgICAgICAgICAgICAjIHRvdGFsIGZyYW1lcyB0byBnZW5lcmF0ZQpjb250ZXh0X2xlbmd0aDogMSAgICAgICAjIHVzZSBvbmx5IHRoZSBmaXJzdCBmcmFtZSBhcyBjb250ZXh0CmZyYW1lX3NraXA6IDEgICAgICAgICAgICMgbm8gdGVtcG9yYWwgc2tpcCBmb3IgY3VzdG9tIHRyYWplY3RvcmllcwoKIyDilIDilIAgTWlzYyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKc2F2ZV9kaXI6IC90bXAvc2luZ2xlX2ltYWdlX3RyYWplY3RvcnkgICMgdW51c2VkIChubyBkYXRhIGRvd25sb2FkKQpmaWx0ZXJfbWluX2xlbjogbnVsbApzdWJkYXRhc2V0X3NpemU6IG51bGwKbnVtX2V2YWxfdmlkZW9zOiAxCm51bV9zYW1wbGVzOiAxICAgICAgICAgICMgaG93IG1hbnkgaW5kZXBlbmRlbnQgdmlkZW9zIHRvIGdlbmVyYXRlIHBlciBydW4K"
)
target = pathlib.Path(REPO_DIR) / 'configurations/dataset/single_image_trajectory.yaml'
target.write_bytes(base64.b64decode(CONFIG_B64))

# ── 3. generate_video.py ───────────────────────────────────────────────────────
GENERATE_B64 = (
    "..."
)
target = pathlib.Path(REPO_DIR) / 'generate_video.py'
target.write_bytes(base64.b64decode(GENERATE_B64))

# ── 4. Patch datasets/video/__init__.py ───────────────────────────────────────
init_path = pathlib.Path(REPO_DIR) / 'datasets/video/__init__.py'
init_text = init_path.read_text()
if 'SingleImageTrajectoryDataset' not in init_text:
    init_path.write_text(init_text.rstrip() + '\nfrom .single_image_trajectory import SingleImageTrajectoryDataset\n')

# ── 5. Patch experiments/video_generation.py ─────────────────────────────────
exp_path = pathlib.Path(REPO_DIR) / 'experiments/video_generation.py'
exp_text = exp_path.read_text()
if 'SingleImageTrajectoryDataset' not in exp_text:
    exp_text = exp_text.replace(
        'from datasets.video import (',
        'from datasets.video import (\n    SingleImageTrajectoryDataset,'
    )
    exp_text = exp_text.replace(
        'kinetics_600=Kinetics600AdvancedVideoDataset,',
        'kinetics_600=Kinetics600AdvancedVideoDataset,\n        single_image_trajectory=SingleImageTrajectoryDataset,'
    )
    exp_path.write_text(exp_text)

print("All custom files written and patches applied.")


In [ ]:
import os, subprocess, sys
REPO_DIR = '/content/diffusion-forcing-transformer'
result = subprocess.run(
    [sys.executable, '-c',
     'import sys; sys.path.insert(0,".")'
     '; from datasets.video.single_image_trajectory import SingleImageTrajectoryDataset'
     '; print("SingleImageTrajectoryDataset OK")'
     '; from experiments.video_generation import VideoGenerationExperiment'
     '; assert "single_image_trajectory" in VideoGenerationExperiment.compatible_datasets'
     '; print("VideoGenerationExperiment registration OK")'
    ],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout or result.stderr)
if result.returncode != 0:
    print("ERROR:", result.stderr)


## Section 2 — Configure (edit this section)

In [ ]:
import os

# ─────────────────────────────────────────────────────────────────────────────
# EDIT THESE TWO PATHS
# Upload your files to Google Drive and paste the paths below.
# ─────────────────────────────────────────────────────────────────────────────
IMAGE_PATH      = '/content/drive/MyDrive/my_image.jpg'        # <-- your image
TRAJECTORY_PATH = '/content/drive/MyDrive/my_trajectory.txt'   # <-- your .txt

# ─────────────────────────────────────────────────────────────────────────────
# Output location (auto-created inside Google Drive)
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = '/content/drive/MyDrive/DFoT_Outputs'
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'generated_video.mp4')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Generation settings
# ─────────────────────────────────────────────────────────────────────────────
N_FRAMES       = 50      # total frames to generate (more = longer runtime)
FPS            = 10      # output video frame rate
GUIDANCE_SCALE = 4.0     # higher = stronger conditioning (3–7 typical)
GUIDANCE_TYPE  = 'stabilized_vanilla'   # 'vanilla' | 'stabilized_vanilla' | 'temporal'
CHECKPOINT     = 'pretrained:DFoT_RE10K.ckpt'   # auto-downloads first time

# ─────────────────────────────────────────────────────────────────────────────
# Validate paths
# ─────────────────────────────────────────────────────────────────────────────
assert os.path.isfile(IMAGE_PATH),      f"Image not found: {IMAGE_PATH}"
assert os.path.isfile(TRAJECTORY_PATH), f"Trajectory not found: {TRAJECTORY_PATH}"
print(f"Image      : {IMAGE_PATH}")
print(f"Trajectory : {TRAJECTORY_PATH}")
print(f"Output     : {OUTPUT_PATH}")
print(f"Frames     : {N_FRAMES}  |  FPS: {FPS}  |  Guidance: {GUIDANCE_TYPE} x{GUIDANCE_SCALE}")


## Section 3 — Generate

In [ ]:
%cd /content/diffusion-forcing-transformer

!python generate_video.py \
    --image          "{IMAGE_PATH}" \
    --trajectory     "{TRAJECTORY_PATH}" \
    --output         "{OUTPUT_PATH}" \
    --n_frames       {N_FRAMES} \
    --checkpoint     "{CHECKPOINT}" \
    --guidance_scale {GUIDANCE_SCALE} \
    --guidance_type  {GUIDANCE_TYPE} \
    --fps            {FPS} \
    --device         cuda


In [ ]:
from IPython.display import Video, display
import os

if os.path.isfile(OUTPUT_PATH):
    display(Video(OUTPUT_PATH, width=640, embed=True))
else:
    print("Video not found — check the error output above.")
